<a href="https://colab.research.google.com/github/21centjoe/Synthetic-Quantum-Leads/blob/main/OneDeeOSvisualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>NELOS 1D OS Simulation</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
    <script src="https://unpkg.com/three@0.128.0/examples/js/controls/OrbitControls.js"></script>
    <style>
        body, html { margin: 0; padding: 0; width: 100%; height: 100%; overflow: hidden; background: #000; font-family: 'Inter', sans-serif; }
        #canvas-container { width: 100vw; height: 100vh; display: block; }
        .overlay-header { position: absolute; top: 0; left: 0; width: 100%; z-index: 10; padding: 1.5rem; color: #fff; background: linear-gradient(to bottom, rgba(0,0,0,0.9), transparent); backdrop-filter: blur(4px); }
        .stripe-gum { display: flex; gap: 4px; height: 16px; width: 240px; }
        .segment { flex: 1; height: 100%; transition: background 0.3s; border-radius: 2px; }
        #data-overlay {
            position: fixed;
            top: 50%;
            left: 50%;
            transform: translate(-50%, -50%) scale(0.9);
            width: 70%;
            max-width: 800px;
            max-height: 80%;
            background-color: rgba(26, 32, 44, 0.95); /* Darker, slightly transparent background */
            border: 1px solid rgba(75, 85, 99, 0.6); /* Subtle border */
            border-radius: 12px;
            padding: 2.5rem;
            color: #e2e8f0; /* Light gray text */
            box-shadow: 0 10px 30px rgba(0, 255, 128, 0.2), 0 0 15px rgba(0, 255, 128, 0.1); /* Emerald glow effect */
            z-index: 100;
            opacity: 0;
            pointer-events: none; /* Make it unclickable when hidden */
            transition: opacity 0.3s ease-out, transform 0.3s ease-out;
            display: flex;
            flex-direction: column;
            gap: 1.5rem;
            overflow-y: auto; /* Enable scrolling for content */
            font-family: 'Inter', sans-serif;
        }
        #data-overlay.visible {
            opacity: 1;
            pointer-events: auto;
            transform: translate(-50%, -50%) scale(1);
        }
        #data-overlay h2 {
            font-size: 1.875rem; /* text-3xl */
            font-weight: 700; /* font-bold */
            color: #34d399; /* emerald-400 */
            border-bottom: 2px solid #10b981; /* emerald-500 */
            padding-bottom: 0.75rem;
            margin-bottom: 1.5rem;
        }
        #data-overlay p {
            font-size: 1.125rem; /* text-lg */
            line-height: 1.6;
            color: #cbd5e1; /* slate-300 */
        }
        #overlay-close-btn {
            position: absolute;
            top: 1rem;
            right: 1.5rem;
            background: none;
            border: none;
            font-size: 2rem;
            color: #94a3b8; /* slate-400 */
            cursor: pointer;
            transition: color 0.2s;
        }
        #overlay-close-btn:hover {
            color: #e2e8f0; /* slate-100 */
        }
        #overlay-content {
            display: flex;
            flex-direction: column;
            align-items: center;
            gap: 1rem;
        }
        #overlay-content img {
            max-width: 150px;
            height: auto;
            border-radius: 8px;
            opacity: 0.8;
        }

    </style>
</head>
<body>

    <div class="overlay-header flex items-center justify-between">
        <div class="flex items-center gap-8">
            <h1 class="font-bold text-2xl tracking-widest text-emerald-400">N E L O S <span class="text-white font-light text-xl">1 D O S</span></h1>
            <div class="text-xs uppercase tracking-wider opacity-80 border-l border-white/20 pl-6">
                USING UBUNTU SERVER: <span id="byte-count" class="font-mono text-emerald-300">7,516,192,768</span> BYTES
            </div>
        </div>

        <div class="flex items-center gap-6">
            <div class="text-xs font-mono tracking-widest text-emerald-500" id="ether-status">ETHER ESTABLISHED</div>
            <div class="stripe-gum" id="speedometer">
                <div class="segment bg-yellow-500"></div>
                <div class="segment bg-yellow-500"></div>
                <div class="segment bg-yellow-400"></div>
                <div class="segment bg-orange-400"></div>
                <div class="segment bg-orange-500"></div>
                <div class="segment bg-orange-600"></div>
                <div class="segment bg-red-500"></div>
                <div class="segment bg-red-600"></div>
            </div>
        </div>
    </div>

    <div id="canvas-container"></div>

    <!-- New Data Overlay -->
    <div id="data-overlay">
        <button id="overlay-close-btn">&times;</button>
        <h2>Buckyball Data</h2>
        <div id="overlay-content">
            <img id="overlay-image" src="https://via.placeholder.com/150/00FF80/FFFFFF?text=File" alt="Generic File Icon">
            <p id="overlay-text">this is what this is but each one will do that go all be formatted to this one statement</p>
        </div>
    </div>

    <script>
        let scene, camera, renderer, sphere, buckyballs = [];
        let time = 0;
        let controls;
        let currentMemory = 7516192768; // Initial memory in Bytes
        const memoryGrowthRate = 100000000; // Bytes per frame (adjust as needed)

        // Raycasting variables
        const raycaster = new THREE.Raycaster();
        const mouse = new THREE.Vector2();
        const dataOverlay = document.getElementById('data-overlay');
        const overlayCloseBtn = document.getElementById('overlay-close-btn');
        const overlayImage = document.getElementById('overlay-image');
        const overlayText = document.getElementById('overlay-text');

        // Camera zoom variables
        let clickedBuckyball = null;
        let isZoomedIn = false;
        let isUnzooming = false; // New flag for unzooming
        const zoomDuration = 500; // milliseconds
        let zoomStartTime = 0;
        let reverseZoomStartTime = 0; // New timestamp for reverse zoom
        let initialCameraPosition = new THREE.Vector3();
        let initialCameraLookAt = new THREE.Vector3();
        let currentZoomedInCameraPosition = new THREE.Vector3(); // New variable
        let currentZoomedInCameraLookAt = new THREE.Vector3(); // New variable

        // Define data types
        const dataTypes = ['movie file', 'document file', 'variety of files'];

        // Data for dynamic overlay content
        const dataTypeContent = {
            'movie file': {
                image: 'https://via.placeholder.com/150/FF0000/FFFFFF?text=Movie', // Red for movie
                text: 'This buckyball contains various movie files. Expect to find video streams, codecs, and related metadata within these packets.'
            },
            'document file': {
                image: 'https://via.placeholder.com/150/0000FF/FFFFFF?text=Doc', // Blue for document
                text: 'This buckyball is composed of document files. These packets likely hold text documents, spreadsheets, presentations, or other textual data.'
            },
            'variety of files': {
                image: 'https://via.placeholder.com/150/00FF80/FFFFFF?text=Mixed', // Green for mixed variety
                text: 'This buckyball represents a variety of file types. Its packets could contain a mix of executables, images, audio, configuration files, and more.'
            }
        };

        function formatBytes(bytes) {
            if (bytes === 0) return '0 Bytes';
            const k = 1024;
            const dm = 2; // Decimal places
            const sizes = ['Bytes', 'KB', 'MB', 'GB', 'TB', 'PB', 'EB', 'ZB', 'YB'];
            const i = Math.floor(Math.log(bytes) / Math.log(k));
            return parseFloat((bytes / Math.pow(k, i)).toFixed(dm)) + ' ' + sizes[i];
        }

        function init() {
            scene = new THREE.Scene();
            camera = new THREE.PerspectiveCamera(75, window.innerWidth / window.innerHeight, 0.1, 1000);
            renderer = new THREE.WebGLRenderer({ antialias: true, alpha: true });
            renderer.setSize(window.innerWidth, window.innerHeight);
            document.getElementById('canvas-container').appendChild(renderer.domElement);

            // Create Giant Geodesic Sphere
            const geometry = new THREE.IcosahedronGeometry(2, 4);
            const material = new THREE.MeshBasicMaterial({ color: 0x333333, wireframe: true, transparent: true, opacity: 0.5 });
            sphere = new THREE.Mesh(geometry, material);
            scene.add(sphere);

            // Add ambient light
            const light = new THREE.AmbientLight(0xffffff, 1);
            scene.add(light);

            camera.position.z = 6;

            // Add OrbitControls
            controls = new THREE.OrbitControls(camera, renderer.domElement);
            controls.enableDamping = true;
            controls.dampingFactor = 0.05;
            controls.enabled = true; // Enable controls initially

            // Add event listener for clicks
            window.addEventListener('click', onDocumentClick, false);
            overlayCloseBtn.addEventListener('click', hideDataOverlay, false);

            animate();
        }

        function onDocumentClick(event) {
            if (isZoomedIn || isUnzooming) return; // Prevent new clicks if already zoomed in or unzooming

            // Calculate mouse position in normalized device coordinates (-1 to +1) for both components
            mouse.x = (event.clientX / window.innerWidth) * 2 - 1;
            mouse.y = - (event.clientY / window.innerHeight) * 2 + 1;

            // Update the raycaster with the camera and mouse position
            raycaster.setFromCamera(mouse, camera);

            // Calculate objects intersecting the ray. We now check the first child (the mesh) of each group.
            const intersects = raycaster.intersectObjects(buckyballs.map(group => group.children[0]));

            if (intersects.length > 0) {
                // The clicked object is the mesh, but we need its parent group
                clickedBuckyball = intersects[0].object.parent;
                isZoomedIn = true;
                zoomStartTime = Date.now();
                initialCameraPosition.copy(camera.position);
                initialCameraLookAt.copy(controls.target); // Store current lookAt target

                // Hide the data overlay initially, it will be shown after zoom
                dataOverlay.classList.remove('visible');

                // Disable controls during zoom
                controls.enabled = false;
            } else {
                // If clicked outside buckyball, hide overlay if visible and not zoomed in
                if (dataOverlay.classList.contains('visible')) {
                    hideDataOverlay(); // This will now also handle unzooming if applicable
                }
            }
        }

        function showDataOverlay() {
            if (clickedBuckyball && clickedBuckyball.userData.dataType) {
                const dataType = clickedBuckyball.userData.dataType;
                const content = dataTypeContent[dataType];
                if (content) {
                    overlayImage.src = content.image;
                    overlayText.textContent = content.text;
                }
            }
            dataOverlay.classList.add('visible');
        }

        function hideDataOverlay() {
            dataOverlay.classList.remove('visible');

            if (isZoomedIn) { // If currently zoomed in, initiate unzoom
                isUnzooming = true;
                reverseZoomStartTime = Date.now();
                currentZoomedInCameraPosition.copy(camera.position); // Store current camera state for reverse zoom
                currentZoomedInCameraLookAt.copy(controls.target); // Store current lookAt target for reverse zoom
            } else { // If not zoomed in, just reset state (e.g., if overlay was shown without zoom)
                controls.enabled = true;
                if (clickedBuckyball) {
                    clickedBuckyball.getObjectByName('buckyballMesh').visible = true;
                    clickedBuckyball.getObjectByName('filePacketsGroup').visible = false;
                }
                clickedBuckyball = null;
                isZoomedIn = false;
                isUnzooming = false;
            }
        }

        // New function to create file packets
        function createFilePackets(count, dataType) {
            const packetsGroup = new THREE.Group();
            let packetGeometry;
            let packetMaterial;

            switch (dataType) {
                case 'movie file':
                    packetGeometry = new THREE.SphereGeometry(0.04, 8, 8); // Small spheres
                    packetMaterial = new THREE.MeshBasicMaterial({ color: 0xff0000 }); // Red for movies
                    break;
                case 'document file':
                    packetGeometry = new THREE.BoxGeometry(0.05, 0.05, 0.02); // Flat boxes
                    packetMaterial = new THREE.MeshBasicMaterial({ color: 0x0000ff }); // Blue for documents
                    break;
                case 'variety of files':
                    // Mix of geometries and colors for variety
                    packetGeometry = Math.random() < 0.5 ? new THREE.TetrahedronGeometry(0.05) : new THREE.CylinderGeometry(0.03, 0.03, 0.08, 6);
                    packetMaterial = new THREE.MeshBasicMaterial({ color: new THREE.Color(Math.random(), Math.random(), Math.random()) }); // Random colors
                    break;
                default:
                    packetGeometry = new THREE.BoxGeometry(0.05, 0.05, 0.05);
                    packetMaterial = new THREE.MeshBasicMaterial({ color: 0x00ff00 }); // Default green
            }

            for (let i = 0; i < count; i++) {
                const packet = new THREE.Mesh(packetGeometry, packetMaterial);
                // Position packets randomly around the buckyball's origin
                const offsetRadius = 0.3;
                const phi = Math.random() * Math.PI * 2;
                const theta = Math.random() * Math.PI;
                packet.position.x = offsetRadius * Math.sin(theta) * Math.cos(phi);
                packet.position.y = offsetRadius * Math.sin(theta) * Math.sin(phi);
                packet.position.z = offsetRadius * Math.cos(theta);
                packetsGroup.add(packet);
            }
            return packetsGroup;
        }

        function createBuckyball() {
            const buckyballGroup = new THREE.Group();

            // Assign a random data type
            const randomDataType = dataTypes[Math.floor(Math.random() * dataTypes.length)];
            buckyballGroup.userData.dataType = randomDataType;

            // Original buckyball mesh
            const geom = new THREE.IcosahedronGeometry(0.2, 0);
            const mat = new THREE.MeshBasicMaterial({ color: 0x10b981, wireframe: true });
            const buckyballMesh = new THREE.Mesh(geom, mat);
            buckyballMesh.name = 'buckyballMesh'; // Name for easy identification
            buckyballGroup.add(buckyballMesh);

            // File packets, initially hidden, created based on data type
            const filePacketsGroup = createFilePackets(10, buckyballGroup.userData.dataType); // Pass data type
            filePacketsGroup.name = 'filePacketsGroup'; // Name for easy identification
            filePacketsGroup.visible = false; // Initially hidden
            buckyballGroup.add(filePacketsGroup);

            // Random spawning position
            const phi = Math.random() * Math.PI * 2;
            const theta = Math.random() * Math.PI;
            const radius = 5;
            buckyballGroup.position.x = radius * Math.sin(theta) * Math.cos(phi);
            buckyballGroup.position.y = radius * Math.sin(theta) * Math.sin(phi);
            buckyballGroup.position.z = radius * Math.cos(theta);

            // Store target position on sphere (slightly further out)
            buckyballGroup.userData.targetRadius = 2.2;
            buckyballGroup.userData.speed = 0.02 + Math.random() * 0.03;

            scene.add(buckyballGroup);
            buckyballs.push(buckyballGroup);
            return buckyballGroup; // Return the group
        }

        function updateSpeedometer() {
            const speed = Math.abs(Math.sin(Date.now() * 0.001)) * 0.8 + 0.1;
            const segments = document.querySelectorAll('.segment');
            segments.forEach((seg, i) => {
                const threshold = i / segments.length;
                seg.style.opacity = speed > threshold ? '1' : '0.15';
            });
        }

        function updateMemoryDisplay() {
            currentMemory += memoryGrowthRate; // Increase memory
            const byteCountElement = document.getElementById('byte-count');
            if (byteCountElement) {
                byteCountElement.textContent = formatBytes(currentMemory);
            }
        }

        function animate() {
            requestAnimationFrame(animate);
            time += 0.01;

            if (isZoomedIn && clickedBuckyball && !isUnzooming) {
                const elapsed = Date.now() - zoomStartTime;
                const progress = Math.min(elapsed / zoomDuration, 1);
                const easedProgress = 0.5 - Math.cos(progress * Math.PI) / 2; // Ease-in-out

                // Calculate target camera position (slightly in front of buckyball)
                const targetPosition = clickedBuckyball.position.clone().normalize().multiplyScalar(2.5); // Adjust 2.5 for zoom level

                // Interpolate camera position
                camera.position.lerpVectors(initialCameraPosition, targetPosition, easedProgress);

                // Interpolate camera look-at target
                const targetLookAt = clickedBuckyball.position.clone();
                controls.target.lerpVectors(initialCameraLookAt, targetLookAt, easedProgress);

                // Update controls to reflect new target
                controls.update();

                if (progress === 1) {
                    // Zoom completed
                    showDataOverlay();
                    clickedBuckyball.getObjectByName('buckyballMesh').visible = false;
                    clickedBuckyball.getObjectByName('filePacketsGroup').visible = true;
                }
            } else if (isUnzooming) { // Handle reverse zoom
                const elapsed = Date.now() - reverseZoomStartTime;
                const progress = Math.min(elapsed / zoomDuration, 1);
                const easedProgress = 0.5 - Math.cos(progress * Math.PI) / 2; // Ease-in-out

                // Interpolate camera position back to initial position
                camera.position.lerpVectors(currentZoomedInCameraPosition, initialCameraPosition, easedProgress);

                // Interpolate camera look-at target back to initial look-at
                controls.target.lerpVectors(currentZoomedInCameraLookAt, initialCameraLookAt, easedProgress);

                controls.update();

                if (progress === 1) {
                    // Unzoom completed
                    isUnzooming = false;
                    isZoomedIn = false;
                    if (clickedBuckyball) {
                        clickedBuckyball.getObjectByName('buckyballMesh').visible = true;
                        clickedBuckyball.getObjectByName('filePacketsGroup').visible = false;
                    }
                    clickedBuckyball = null; // Reset clicked buckyball
                    controls.enabled = true; // Re-enable controls
                }
            } else {
                // Update controls only if not zooming or unzooming
                controls.update();
            }

            // Rotate the giant core
            sphere.rotation.y += 0.002;
            sphere.rotation.z += 0.001;

            // Occasionally spawn data units
            if (Math.random() < 0.03) createBuckyball();

            // Handle buckyball movement (only if not zoomed in and focused on one)
            if (!isZoomedIn && !isUnzooming) {
                buckyballs.forEach((buckyballGroup) => {
                    // Only animate if the buckyball mesh is visible (i.e., not zoomed in or unzooming)
                    if (buckyballGroup.getObjectByName('buckyballMesh').visible) {
                        // Move towards sphere surface
                        const dist = buckyballGroup.position.length();
                        if (dist > buckyballGroup.userData.targetRadius) {
                            buckyballGroup.position.multiplyScalar(0.98);
                        } else {
                            // Lock into rotation and orbit
                            buckyballGroup.position.setLength(buckyballGroup.userData.targetRadius); // Ensure it's on the surface
                            // Apply rotations to the mesh inside the group, if needed, or rotate the group itself
                            buckyballGroup.rotation.x += 0.02; // Local rotation for the group
                            buckyballGroup.rotation.y += 0.02; // Local rotation for the group

                            // Orbit the central sphere around its Y-axis
                            buckyballGroup.position.applyAxisAngle(new THREE.Vector3(0, 1, 0), 0.005); // Adjust orbital speed as needed
                        }
                    }
                });
            } else if (clickedBuckyball && clickedBuckyball.getObjectByName('filePacketsGroup').visible) {
                // Rotate the file packets if zoomed in
                clickedBuckyball.getObjectByName('filePacketsGroup').rotation.x += 0.01;
                clickedBuckyball.getObjectByName('filePacketsGroup').rotation.y += 0.01;
            }

            updateSpeedometer();
            updateMemoryDisplay(); // Update memory display in each frame
            renderer.render(scene, camera);
        }

        window.addEventListener('resize', () => {
            camera.aspect = window.innerWidth / window.innerHeight;
            camera.updateProjectionMatrix();
            renderer.setSize(window.innerWidth, window.innerHeight);
        });

        init();
    </script>
</body>
</html>

# Task
The user wants to enhance an existing Three.js 1D OS simulation. The simulation currently displays a central sphere and 'buckyballs' moving around it, along with a memory display and a speedometer. The main goal is to add interactivity by allowing users to click on the 'buckyballs' to reveal an informational overlay. This involves implementing raycasting for click detection, creating a dynamic and styled HTML overlay, animating its appearance and dismissal, and populating it with placeholder data and text. The overall objective is to create a more engaging and informative user experience within the simulation.

## Implement Raycasting for Buckyball Clicks

### Subtask:
Set up a Three.js Raycaster to enable click detection on the buckyball objects in the 3D scene, identifying which buckyball is selected.

## Create Dynamic & Styled Data Overlay

### Subtask:
Develop a new HTML 'div' element that will function as an informational overlay. This overlay will be styled with CSS to be initially hidden and prepared for dynamic content and smooth animations, including a subtle 'zoom' effect.

## Animate Overlay Appearance for 'Focus'

### Subtask:
Implement the JavaScript logic to reveal the informational overlay with a smooth fade-in and scale-up animation when a buckyball is clicked.

## Display Placeholder Data and Text

### Subtask:
Populate the active overlay with a generic image (representing a document or folder) and the specified explanatory text.

## Implement Animated Overlay Dismissal

### Subtask:
Add a mechanism (e.g., a close button within the overlay or an event listener for clicks outside) to smoothly animate the overlay's disappearance (e.g., fade-out and scale-down) when the user wishes to return to the main visualization.

# Task
The user wants to enhance an existing Three.js 1D OS simulation. The simulation currently displays a central sphere and 'buckyballs' moving around it, along with a memory display and a speedometer. The main goal is to add interactivity by allowing users to click on the 'buckyballs' to reveal an informational overlay. This involves implementing raycasting for click detection, creating a dynamic and styled HTML overlay, animating its appearance and dismissal, and populating it with placeholder data and text. The overall objective is to create a more engaging and informative user experience within the simulation.

## Detect Buckyball Click and Initiate Camera Zoom

### Subtask:
Modify the click handling to identify the clicked buckyball and smoothly animate the Three.js camera to move towards and focus on the selected buckyball, providing a continuous 'zoom in' effect.

## Transform Buckyball to Display Internal 'File Packets'

### Subtask:
Upon reaching the focused buckyball, visually transform its geometry or replace it with a new set of smaller 3D objects representing 'file packets'. These packets should appear to emerge from or be contained within the original buckyball structure.

## Assign and Visualize Different Data Types

### Subtask:
Implement logic to randomly assign one of three data types ('movie file', 'document file', 'variety of files') to the internal 'file packets' of each buckyball. Visually differentiate these packet types (e.g., distinct colors, textures, or subtle geometric variations) to enhance the 'random' and 'different things' aspect.

## Populate Dynamic Information Overlay for Packet Data

### Subtask:
Enhance the existing HTML data overlay to dynamically populate it with relevant images and specific explanatory text that corresponds to the buckyball's assigned data type when focused on a buckyball's 'file packets'.

## Implement Smooth Reverse Zoom and State Reset

### Subtask:
Create a mechanism to smoothly animate the camera back to its initial overview position and revert the 'file packets' visualization back to the original buckyball geometry, resetting the simulation state.